In [1]:
import io
import re
from datetime import datetime

import numpy as np
import pandas as pd
import requests
from IPython.display import Markdown, display

# -------------------------
# Configuración / constantes
# -------------------------

SHEET_ID = "1WQ5eKmttF25JonqEcwyKcd28PBfmnb16"
SHEET_NAME = "General"

URL = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=xlsx"


weight_category_map = {
    "DD": 0,
    "NE": 0,
    "LC": 0,
    "NT": 1,
    "VU": 2,
    "EN": 3,
    "CR": 4,
    "CO": 5,
}

M = 5  # Peso máximo para el cálculo del RLI

def compute_rli(group: pd.DataFrame) -> pd.Series:
    """Calcula el RLI sobre un grupo de filas (año, o global)."""
    N = len(group)
    sum_w = group["weight_category"].sum()
    rli = 1 - (sum_w / (N * M)) if N > 0 else np.nan
    return pd.Series(
        {
            "Número total de ecosistemas (pisos vegetacionales) evaluados.": N,
            "suma_pesos": sum_w,
            "RLI": rli,
        }
    )


# -------------------------
# Descarga y preparación del DataFrame
# -------------------------

resp = requests.get(URL)
resp.raise_for_status()

df = pd.read_excel(
    io.BytesIO(resp.content),
    sheet_name=SHEET_NAME,
)

df["criterio_final_uicn"] = df["Criterio final de UICN"]

# -------------------------
# Codificar categorías y pesos
# -------------------------
# Excluir DD y R
df = df[~df["criterio_final_uicn"].isin(["DD", "NE", "LC", ""])]
df["weight_category"] = df["criterio_final_uicn"].map(weight_category_map).astype("Int64")
# Eliminar filas sin categoría o sin peso
df = df.dropna(subset=["criterio_final_uicn", "weight_category"])

# -------------------------
# Cálculo del RLI
# -------------------------

rli_global = compute_rli(df)
print("RLI global:")
print(rli_global)

RLI global:
Número total de ecosistemas (pisos vegetacionales) evaluados.    19.000000
suma_pesos                                                       64.000000
RLI                                                               0.326316
dtype: float64


# A.1: Lista Roja de Ecosistemas (Red List of Ecosystems)

## Instrucciones

### Acceso a metadatos

- Acceso a metadatos: https://www.gbf-indicators.org/metadata/headline/A-1

## 1. Definición y Objetivo

Este indicador evalúa el riesgo de colapso de los ecosistemas del país. Mide qué tan cerca están los ecosistemas de perder su distribución natural o función ecológica.

- **Métrica de reporte:** Índice de la Lista Roja de Ecosistemas (**RLIe**).
- **Escala del índice:** valor decimal de **0 a 1**.
  - **1** = Todos los ecosistemas clasificados como **“Preocupación Menor”** (intactos).
  - **0** = Todos los ecosistemas han **“Colapsado”**.
- **Unidad de evaluación nacional:** Pisos Vegetacionales (Luebert & Pliscoff).
- **Unidad de agregación global:** Grupos Funcionales de Ecosistemas (**EFG** - Nivel 3 UICN).

---

## 2. Fuente de Datos Oficial

Para el cálculo, se debe utilizar la **clasificación IUCN de los pisos vegetacionales**, correspondientes a datos validados por el Ministerio del Medio Ambiente.

- **Repositorio:** Sistema de Información de Biodiversidad (**SIMBIO**).
- **URL:** https://simbio.mma.gob.cl/Ecosistemas

### Procedimiento de descarga

1. Ingresar al enlace.
2. Hacer clic en el botón **“Descargar catálogo extendido”** (formato Excel/CSV).
3. Variable clave: columna **“Criterio final de UICN”** (resultado consolidado de la evaluación: CR, EN, VU, etc.).

---

## 3. Metodología de Cálculo

El cálculo se basa en transformar las categorías cualitativas en puntajes cuantitativos para aplicar la fórmula del índice **RLIe**.

### Paso 1: Preparación de la base de datos

- Abrir la planilla descargada de SIMBIO.
- Filtrar o eliminar registros **“No Evaluado” (NE)** o **“Datos Insuficientes” (DD)**, ya que no aportan al cálculo del riesgo.

### Paso 2: Asignación de puntajes (pesos)

Crear una columna llamada **`Puntaje_Riesgo`** y asignar valores numéricos según la columna **“Criterio final de UICN”**:

| Categoría UICN (Sigla) | Descripción              | Puntaje (Pi) |
|---|---|---:|
| LC | Preocupación Menor         | 0 |
| NT | Casi Amenazado             | 1 |
| VU | Vulnerable                 | 2 |
| EN | En Peligro                 | 3 |
| CR | En Peligro Crítico         | 4 |
| CO | Colapsado                  | 5 |

### Paso 3: Fórmula del índice (RLIe)

$$
RLIe = 1 - \frac{\sum P_i}{M \times N}
$$

Donde:

- \(\sum P_i\): suma total de los puntajes de todos los ecosistemas evaluados.  
- \(M\): puntaje máximo posible (**constante = 5**).  
- \(N\): número total de ecosistemas (pisos vegetacionales) evaluados.

---

## 4. Instrucciones para el Reporte (Llenado de Plantilla)

Debes llenar la plantilla CSV/Excel del indicador A.1 reportando primero el valor nacional y luego desagregando por grupos funcionales.

### A. Nivel Nacional (Fila obligatoria)

Calcula el **RLIe** usando todos los pisos vegetacionales de Chile continental.

- **Year:** [Año de la evaluación, ej. 2017]  
- **Unit:** Index (0-1)  
- **Value:** [Resultado de la fórmula]  
- **Disaggregation type:** Total (o dejar en blanco según plantilla)

### B. Desagregación por EFG (Filas adicionales)

Repetir el cálculo de la fórmula, pero filtrando la base de datos por cada **Grupo Funcional (EFG)**.

#### Ejemplo práctico: EFG “T3.2” (Bosques Esclerófilos)

1. Filtrar Excel para mostrar solo los pisos vegetacionales clasificados como **T3.2**. Supongamos que son **15** pisos.
2. Sumar los pesos solo de esos 15 pisos. (Supongamos suma **50**).
3. Calcular el máximo posible para ese grupo: \(15 \times 5 = 75\).
4. Aplicar la fórmula:

$$
RLIe = 1 - \frac{50}{75} = 1 - 0.66 = 0.34
$$

**Resultado desagregado:** El RLIe para el EFG **T3.2** es **0.34** (valor bajo = alto riesgo de colapso).

---

## Ejemplo de reporte

| Nivel de Reporte | Total Evaluados (N) | RLIe (0 a 1) | Interpretación del Estado |
|---|---:|---:|---|
| Nacional (Todos los Pisos) | 127 | 0.65 (ejemplo) | Estado medio. Riesgo moderado de colapso a nivel país. |
| T3.2 Bosques y Matorrales Esclerófilos | 25 | 0.34 | Estado crítico. Alta concentración de ecosistemas CR y EN. |
| T2.2 Bosques Templados Caducifolios | 12 | 0.55 | Riesgo moderado a alto (amenazas por uso de suelo). |
| T2.3 Bosques Templados Siempreverdes | 30 | 0.85 | Buen estado. Mayoría en categoría LC o NT. |
| T5.2 Desiertos Suculentos (Atacama) | 18 | 0.78 | Buen estado general, amenazas localizadas. |


## Resultados
### Nivel Nacional

In [2]:
print(f'VALOR RLI 2026: {rli_global["RLI"]}')

VALOR RLI 2026: 0.3263157894736842


In [5]:
df.to_excel("A_1_rli_ecosystem.xlsx")

In [6]:
import geopandas as gpd
from sqlalchemy import create_engine
import os
def make_engine_from_env(host_override=None):
    host = host_override or os.getenv("POSTGRES_HOST", "localhost")
    port = os.getenv("POSTGRES_PORT", "5432")
    user = os.getenv("POSTGRES_USER", "postgres")
    password = 'super_secret_pass'#os.getenv("POSTGRES_PASSWORD", "")
    db = os.getenv("POSTGRES_DB", "postgres")

    # OJO: en .env no pongas comillas dentro del valor. Debería ser sin ''.
    # Si ya tienes comillas en el valor, aquí igual funcionará en la mayoría de casos,
    # pero lo correcto es quitarlas del archivo.
    password = password.strip("'").strip('"')

    url = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}"
    return create_engine(url, pool_pre_ping=True)

engine = make_engine_from_env(host_override="db")  # desde tu host
efg_continental_gpd = gpd.read_postgis(
    sql="SELECT * FROM public.efg_continental",
    con=engine,
    geom_col="geometry",     
    crs="EPSG:4326"      
)

DETAIL:  The database was created using collation version 2.41, but the operating system provides version 2.31.
HINT:  Rebuild all objects in this database that use the default collation and run ALTER DATABASE postgres REFRESH COLLATION VERSION, or build PostgreSQL with the right library version.


In [7]:
efg_continental_df = pd.DataFrame(efg_continental_gpd)

In [16]:
mask = df["Nombre"] == "Bosque esclerofilo mediterraneo andino de Lithrea caustica - Lomatia hirsuta"
df.loc[mask, "Nombre"] = "Bosque esclerofilo mediterráneo andino de Lithrea caustica - Lomatia hirsuta"

mask = df["Nombre"] == "Bosque esclerofilo psamófilo mediterráneo interior de Quillaja saponaria/Fabiana imbricata"
df.loc[mask, "Nombre"] = "Bosque esclerofilo psamófilo mediterráneo interior de Quillaja saponaria / Fabiana imbricata"

mask = df["Nombre"] == "Bosque caducifolio mediterráneo costero de Nothofagus glauca y Persea lingue"
df.loc[mask, "Nombre"] = "Bosque caducifolio mediterráneo costero de Nothofagus glauca - Persea lingue"

mask = df["Nombre"] == "Bosque caducifolio mediterráneo andino de Nothofagus glauca - Nothofagus obliqua"
df.loc[mask, "Nombre"] = "Bosque caducifolio mediterráneo andino de Nothofagus glauca - N. obliqua"

mask = df["Nombre"] == "Bosque caducifolio mediterráneo-templado costero de Nothofagus obliqua - Gomortega keule"
df.loc[mask, "Nombre"] = "Bosque caducifolio mediterráneo costero de Nothofagus obliqua - Gomortega keule"

mask = df["Nombre"] == "Bosque caducifolio templado de Nothofagus obliqua - Persea lingue"
df.loc[mask, "Nombre"] = "Bosque caducifolio mediterráneo de Nothofagus obliqua - Persea lingue"

mask = df["Nombre"] == "Bosque mixto templado costero de Nothofagus dombeyi - Nothofagus obliqua"
df.loc[mask, "Nombre"] = "Bosque mixto mediterráneo-templado costero de Nothofagus dombeyi - N. obliqua"

In [17]:
df_join = df.merge(
    efg_continental_df,
    how="left",
    left_on="Nombre",      # columnas en df1
    right_on="PISO"      # columnas en df2 (ajusta si tienen otro nombre)
)

In [19]:
df_join.to_excel("A_1_rli_ecosystem_join_efg_continental.xlsx")

In [27]:
df_join["FORMACION"] = df_join["FORMACION"].astype(str).str.strip()

rli_por_formacion = (
    df_join.groupby("FORMACION", dropna=False)
      .apply(compute_rli)
      .reset_index()
      .sort_values("RLI", ascending=True)  # o False si quieres mejores primero
)

### Desagregación por Formación

In [28]:
tabla_md = (
    rli_por_formacion
    .loc[:, ["FORMACION", "RLI"]]
    .sort_values("RLI", ascending=True)
)

# (opcional) redondear para que se vea lindo
tabla_md["RLI"] = tabla_md["RLI"].round(4)

md = tabla_md.to_markdown(index=False)
display(Markdown(md))

| FORMACION          |   RLI |
|:-------------------|------:|
| Bosque laurifolio  |  0.2  |
| Bosque caducifolio |  0.26 |
| Bosque esclerofilo |  0.3  |
| Bosque espinoso    |  0.55 |

### Desagregación por EFG

In [35]:
df_join["Group_"] = df_join["Group_"].astype(str).str.strip()

rli_por_piso = (
    df_join.groupby("Group_", dropna=False)
      .apply(compute_rli)
      .reset_index()
      .sort_values("RLI", ascending=True)  # o False si quieres mejores primero
)

In [36]:
tabla_md = (
    rli_por_piso
    .loc[:, ["Group_", "RLI"]]
    .sort_values("RLI", ascending=True)
)

# (opcional) redondear para que se vea lindo
tabla_md["RLI"] = tabla_md["RLI"].round(4)

md = tabla_md.to_markdown(index=False)
display(Markdown(md))

| Group_                                        |    RLI |
|:----------------------------------------------|-------:|
| Oceanic cool temperate rainforests            | 0.2545 |
| Seasonally dry temperate heath and shrublands | 0.3    |
| Tropical/Subtropical dry forests and thickets | 0.55   |